# Trích xuất Text Embeddings cho Đồ án Tốt nghiệp (DATN)

Notebook chạy trên Kaggle (GPU T4/P100) để trích xuất đặc trưng văn bản sản phẩm bằng mô hình đa phương thức **Jina CLIP v2** (`jinaai/jina-clip-v2`, output 1024 chiều, hỗ trợ đa ngôn ngữ, Matryoshka truncation).

## 1. Cài đặt thư viện

Kaggle có sẵn `transformers` nhưng bản cài sẵn có bug khi nạp Jina CLIP v2 (so sánh nhầm `str` với `int` lúc sort state_dict), nên mình ghim cứng về bản `5.3.0` đã test chạy ổn.

In [ ]:
!pip install -q --upgrade "transformers==5.3.0" polars pyarrow torch einops timm


## 2. Import thư viện & kiểm tra GPU

In [ ]:
import os
import numpy as np
import polars as pl
import torch
from transformers import AutoModel
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


## 3. Cấu hình đường dẫn dữ liệu

Ưu tiên chạy trên Kaggle (`/kaggle/input/...`), có fallback sang `../data` để test nhanh ở local trước khi submit.

In [ ]:
INPUT_DIR = "/kaggle/input/datn-stream-subset"
OUTPUT_DIR = "/kaggle/working"

MODEL_NAME = "jinaai/jina-clip-v2"

items_path = os.path.join(INPUT_DIR, "items.parquet")
if not os.path.exists(items_path):
    # fallback để test nhanh ở local
    items_path = "../data/items.parquet"
    INPUT_DIR = "../data"

print(f"Loading items from: {items_path}")


## 4. Đọc dữ liệu sản phẩm

In [ ]:
df_items = pl.read_parquet(items_path)
print(f"Tổng số sản phẩm: {len(df_items)}")
print("Các cột dữ liệu:", df_items.columns)
df_items.head(3)


## 5. Tiền xử lý văn bản

Gộp `title`, `brand`, `description`, `features` thành một chuỗi đại diện cho mỗi sản phẩm, để đưa vào model dưới dạng 1 đoạn text duy nhất.

In [ ]:
def prepare_text(row):
    parts = []
    if row["title"]:
        parts.append(f"Title: {row['title']}")
    if row["brand"]:
        parts.append(f"Brand: {row['brand']}")
    if row["description"]:
        parts.append(f"Description: {row['description']}")
    if row["features"]:
        parts.append(f"Features: {row['features']}")

    text = " | ".join(parts).strip()
    return text if text else "unknown"


Điền giá trị rỗng cho các cột null rồi áp dụng `prepare_text` cho toàn bộ dataset.

In [ ]:
df_items = df_items.with_columns([
    pl.col("title").fill_null(""),
    pl.col("brand").fill_null(""),
    pl.col("description").fill_null(""),
    pl.col("features").fill_null(""),
])

items_list = df_items.select(["item_id", "title", "brand", "description", "features"]).to_dicts()
texts_to_encode = [prepare_text(item) for item in tqdm(items_list, desc="Gộp văn bản sản phẩm")]

print(f"Ví dụ văn bản sản phẩm đầu tiên:\n\n{texts_to_encode[0]}")


## 6. Tải mô hình Jina CLIP v2

Bản `transformers` cài sẵn trên Kaggle có 3 lỗi tương thích với Jina CLIP v2, cần vá (monkeypatch) trước khi load model:

1. **`dot_natural_key` so sánh nhầm kiểu dữ liệu** khi sort tên tham số trong state_dict → gây `TypeError: '<' not supported between instances of 'str' and 'int'`.
2. **Thiết bị `meta`**: `transformers` khởi tạo model trên thiết bị "ảo" `meta` trước khi nạp trọng số để load nhanh hơn, nên buffer chưa được tính giá trị thật → phải ép về `cpu`.
3. **Buffer non-persistent bị ghi đè bằng vùng nhớ rác**: sau khi nạp xong, `transformers` vẫn ghi đè các buffer không nằm trong checkpoint (RoPE `freqs_cos/sin`, `inv_freq`...) → phải dựng 1 model tham chiếu rồi copy lại đúng giá trị.

Lỗi thứ 3 nguy hiểm nhất vì không ném exception nào cả — model vẫn chạy nhưng ra vector NaN (hoặc sai lệch âm thầm). Vì vậy cuối phần này luôn có bước "smoke test" để bắt lỗi ngay, trước khi chạy trên toàn bộ dataset.

In [ ]:
def patch_dot_natural_key():
    # Lỗi 1/3: sửa hàm sort dùng khi nạp state_dict, không cho so sánh int với str trực tiếp.
    try:
        import transformers.core_model_loading as _core_model_loading
    except ImportError:
        return False
    if not hasattr(_core_model_loading, "dot_natural_key"):
        return False

    def _safe_dot_natural_key(s):
        return [(0, int(p)) if p.isdigit() else (1, p) for p in s.split(".")]

    _core_model_loading.dot_natural_key = _safe_dot_natural_key
    return True


In [ ]:
import contextlib
import gc
import torch.utils._device as _torch_device_mod
from torch.utils._device import _device_constructors


@contextlib.contextmanager
def meta_init_safe_load():
    # Lỗi 2/3: ép mọi tensor tạo trong lúc khởi tạo model về "cpu" thay vì "meta",
    # để buffer được tính giá trị thật ngay từ đầu.
    orig_call = _torch_device_mod.DeviceContext.__torch_function__

    def patched_call(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        if self.device.type == "meta" and kwargs.get("device") is None and func in _device_constructors():
            kwargs = dict(kwargs)
            kwargs["device"] = "cpu"
            return func(*args, **kwargs)
        return orig_call(self, func, types, args, kwargs)

    _torch_device_mod.DeviceContext.__torch_function__ = patched_call
    try:
        yield
    finally:
        _torch_device_mod.DeviceContext.__torch_function__ = orig_call


In [ ]:
def _iter_non_persistent_buffers(module, prefix=""):
    for name, buf in module._buffers.items():
        if buf is not None and name in module._non_persistent_buffers_set:
            yield (f"{prefix}.{name}" if prefix else name), buf
    for child_name, child in module.named_children():
        child_prefix = f"{prefix}.{child_name}" if prefix else child_name
        yield from _iter_non_persistent_buffers(child, child_prefix)


def restore_non_persistent_buffers(model):
    # Lỗi 3/3 (quan trọng nhất, đã kiểm chứng thực nghiệm): sau from_pretrained,
    # transformers vẫn ghi đè MỌI buffer non-persistent bằng torch.empty_like()
    # (vùng nhớ rác), bất kể patch ở trên đã tính đúng hay chưa.
    # Cách vá: dựng 1 model tham chiếu bằng constructor thẳng (bỏ qua from_pretrained
    # nên không bị ghi đè), rồi copy buffer từ đó sang model thật.
    targets = list(_iter_non_persistent_buffers(model))
    if not targets:
        return []

    with meta_init_safe_load():
        ref_model = type(model)(model.config)
    ref_lookup = dict(_iter_non_persistent_buffers(ref_model))

    restored = []
    for name, buf in targets:
        ref_buf = ref_lookup.get(name)
        if ref_buf is not None and ref_buf.shape == buf.shape:
            buf.data.copy_(ref_buf.data.to(device=buf.device, dtype=buf.dtype))
            restored.append(name)

    del ref_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    missing = [n for n, _ in targets if n not in restored]
    if missing:
        raise RuntimeError(f"Không khôi phục được {len(missing)} buffer: {missing[:10]}")
    return restored


Áp dụng các patch, load model, và khôi phục buffer.

In [ ]:
_patched = patch_dot_natural_key()
print(f"patch_dot_natural_key applied: {_patched}")

print(f"Đang tải mô hình: {MODEL_NAME}...")
with meta_init_safe_load():
    model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float32)
model = model.to(device)
model.eval()
print("Đã tải mô hình thành công.")

_restored = restore_non_persistent_buffers(model)
print(f"Đã khôi phục {len(_restored)} buffer non-persistent (RoPE, ...).")


**Smoke test:** kiểm tra nhanh 1 câu mẫu để chắc chắn model không sinh NaN, trước khi chạy trên toàn bộ dataset.

In [ ]:
with torch.no_grad():
    _smoke_emb = model.encode_text(["smoke test"], convert_to_numpy=True, show_progress_bar=False)
_smoke_emb = np.asarray(_smoke_emb, dtype=np.float32)
assert not np.isnan(_smoke_emb).any(), "Model sinh vector NaN ngay ở bước kiểm tra nhanh - dừng lại, không chạy full dataset!"
print(f"Smoke test OK - vector mẫu không chứa NaN (shape={_smoke_emb.shape}, dtype={_smoke_emb.dtype}).")


## 7. Trích xuất đặc trưng cho toàn bộ dataset

Chạy theo batch. Sau mỗi batch kiểm tra NaN ngay để dừng sớm nếu có lỗi, thay vì chạy hết rồi mới phát hiện embedding hỏng.

In [ ]:
batch_size = 128
all_embeddings = []
item_ids_list = df_items["item_id"].to_list()

print("Đang trích xuất đặc trưng văn bản...")
with torch.no_grad():
    for i in tqdm(range(0, len(texts_to_encode), batch_size), desc="Trích xuất text embeddings"):
        batch_texts = texts_to_encode[i:i + batch_size]
        batch_embs = model.encode_text(batch_texts, convert_to_numpy=True, show_progress_bar=False)
        batch_embs = np.asarray(batch_embs, dtype=np.float32)  # ép float32 tường minh cho cosine similarity sau này

        # Kiểm tra NaN ngay trên từng batch, dừng sớm thay vì lưu hết rồi mới phát hiện lỗi
        nan_mask = np.isnan(batch_embs).any(axis=1)
        if nan_mask.any():
            bad_ids = [item_ids_list[i + j] for j in np.where(nan_mask)[0]]
            raise RuntimeError(f"Phát hiện {int(nan_mask.sum())} vector NaN trong batch [{i}:{i + batch_size}], item_id lỗi: {bad_ids[:10]}")
        all_embeddings.append(batch_embs)

embeddings = np.vstack(all_embeddings).astype(np.float32)
print(f"Kích thước ma trận embeddings: {embeddings.shape}, dtype: {embeddings.dtype}")


## 8. Lưu kết quả

In [ ]:
assert not np.isnan(embeddings).any(), "Phát hiện NaN trong embeddings trước khi lưu!"
assert not np.isinf(embeddings).any(), "Phát hiện Inf trong embeddings trước khi lưu!"
assert embeddings.dtype == np.float32, f"Kiểu dữ liệu không mong đợi: {embeddings.dtype}"

emb_output_path = os.path.join(OUTPUT_DIR, "text_embeddings.npy")
np.save(emb_output_path, embeddings)
print(f"Đã lưu ma trận vector nhúng tại: {emb_output_path}")

df_metadata = pl.DataFrame({
    "index": list(range(len(df_items))),
    "item_id": df_items["item_id"].to_list(),
})
meta_output_path = os.path.join(OUTPUT_DIR, "text_embedding_metadata.parquet")
df_metadata.write_parquet(meta_output_path)
print(f"Đã lưu file ánh xạ metadata tại: {meta_output_path}")


## 9. Kiểm tra lại file đã lưu

In [ ]:
loaded_embeddings = np.load(emb_output_path)
print(f"Kiểm tra kích thước file tải lại: {loaded_embeddings.shape}, dtype: {loaded_embeddings.dtype}")
assert not np.isnan(loaded_embeddings).any(), "File đã lưu chứa NaN!"
assert np.allclose(embeddings, loaded_embeddings), "Dữ liệu lưu bị lỗi!"
print("Kiểm tra hoàn tất: không có NaN, dữ liệu khớp với bản gốc trong bộ nhớ.")
